# Cross-Model Baseline — visualizations (per side model)

Per-round trajectories for the **cross-model baseline** (a MAIN model — Qwen2.5-14B — reads documents
synthesized from a SIDE model's answers; the MAIN model is measured via `run["answer"]`). Three side
models: **DeepSeek-R1-Distill-7B**, **Llama-3.1-8B**, **Mistral-7B**.

Figures are written **per side model** to `cross_model_baseline_visualizations/<side>/`, each overlaying
that side's `replace_all` / `replace_one` / `search` in `visualization.ipynb` style
(Replace All `#1f77b4` / Replace One `#ff7f0e` / Search `#2ca02c`).

- **Text metrics** (all 3 sides, from `evaluation.py`): cosine / ROUGE-L / TES, `same_answer%`,
  `unique_words`, `ai_reference%`, plus a combined grid. Read from
  `cross-model-baseline/evaluation_outputs/<side>/local_<variant>_eval.json` (gitignored downloads).
- **Entity metrics** (DeepSeek side only — the only cross-model entity data in the dump): `unique_entities`,
  `entity_similarity`, `collapse_by_simulation` (95% Wilson CI) for `replace_all`/`replace_one`, from the
  latest `entity re-run for workshop paper-*` dump.

In [1]:
import os
import json
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

# ── Config ──
EVAL_ROOT = "cross-model-baseline/evaluation_outputs"   # side-keyed: <side>/local_<variant>_eval.json
OUT_ROOT  = "cross_model_baseline_visualizations"
SIDES = ["deepseek-r1-distill-qwen-7b", "llama-3.1-8b", "mistral-7b"]

# label -> eval filename (missing files are skipped)
VARIANTS = {
    "Replace All": "local_replace_all_eval.json",
    "Replace One": "local_replace_one_eval.json",
    "Search":      "local_search_eval.json",
}
COLORS = {"Replace All": "#1f77b4", "Replace One": "#ff7f0e", "Search": "#2ca02c"}

# (json_key, title, y-axis label)
METRICS = [
    ("avg_pairwise_similarity", "Cosine similarity across runs", "avg pairwise cosine"),
    ("avg_pairwise_rougeL",     "ROUGE-L across runs",           "avg pairwise ROUGE-L"),
    ("avg_pairwise_tes",        "Token-edit similarity (TES)",   "avg pairwise TES"),
    ("same_answer_percentage",  "Same-answer % (LLM judge)",     "same-answer %"),
    ("unique_words",            "Unique words (round union)",    "unique words"),
    ("ai_reference_percentage", "AI-reference %",                "ai-reference %"),
]

def per_round(experiment, key):
    """Mean of metric `key` across questions per round (1-indexed on the x-axis when plotted)."""
    buckets = defaultdict(list)
    for q in experiment["questions"]:
        for it in q["iterations"]:
            v = it.get("metrics", {}).get(key)
            if v is not None:
                buckets[it["iteration_number"]].append(v)
    return [float(np.mean(buckets[r])) for r in sorted(buckets)]

def load_side(side):
    """Load the available variant eval files for one side model."""
    data = {}
    for label, fname in VARIANTS.items():
        p = os.path.join(EVAL_ROOT, side, fname)
        if os.path.isfile(p):
            with open(p, encoding="utf-8") as f:
                data[label] = json.load(f)
    return data

# ── Text-metric figures, one folder per side (RA/RO/Search overlaid) ──
for side in SIDES:
    data = load_side(side)
    if not data:
        print("skip (no eval files):", side); continue
    outdir = os.path.join(OUT_ROOT, side)
    os.makedirs(outdir, exist_ok=True)
    # one figure per metric
    for key, title, ylabel in METRICS:
        fig, ax = plt.subplots(figsize=(10, 6))
        for label, exp in data.items():
            vals = per_round(exp, key)
            if vals:
                ax.plot(range(1, len(vals) + 1), vals, linewidth=2.5, color=COLORS[label], label=label)
        ax.set_xlabel("Round", fontsize=12)
        ax.set_ylabel(ylabel, fontsize=12)
        ax.set_title(f"{side} — {title}", fontsize=13)
        ax.legend()
        fig.tight_layout()
        fig.savefig(f"{outdir}/{key}_per_round.png", dpi=150, bbox_inches="tight")
        plt.close(fig)
    # combined grid
    ncol = 2
    nrow = (len(METRICS) + ncol - 1) // ncol
    fig, axes = plt.subplots(nrow, ncol, figsize=(14, 5 * nrow))
    axes = np.array(axes).reshape(-1)
    for ax, (key, title, ylabel) in zip(axes, METRICS):
        for label, exp in data.items():
            vals = per_round(exp, key)
            if vals:
                ax.plot(range(1, len(vals) + 1), vals, linewidth=2.5, color=COLORS[label], label=label)
        ax.set_title(title, fontsize=13)
        ax.set_xlabel("Round", fontsize=11)
        ax.set_ylabel(ylabel, fontsize=11)
        ax.legend(fontsize=8)
    for ax in axes[len(METRICS):]:
        ax.set_visible(False)
    fig.suptitle(f"Cross-model baseline ({side}) — text metrics per round", fontsize=15)
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    fig.savefig(f"{outdir}/all_metrics_per_round.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"wrote {side}/: {list(data)}  ({len(METRICS)} metric PNGs + grid)")

wrote deepseek-r1-distill-qwen-7b/: ['Replace All', 'Replace One', 'Search']  (6 metric PNGs + grid)


wrote llama-3.1-8b/: ['Replace All', 'Replace One', 'Search']  (6 metric PNGs + grid)


wrote mistral-7b/: ['Replace All', 'Replace One', 'Search']  (6 metric PNGs + grid)


## Entity-collapse diagrams (DeepSeek side only)

The dump only contains cross-model entity data for the **DeepSeek side**, `replace_all` + `replace_one`
(Qwen main model, GPT-5.2-tagged). Recomputed here (matching `entity_extraction.py` / `visualization.ipynb`)
and written into that side's folder, `cross_model_baseline_visualizations/deepseek-r1-distill-qwen-7b/`:

- `unique_entities_per_round.png` — union of canonical entities across the 10 runs, averaged over questions
- `entity_similarity_per_round.png` — mean pairwise cosine similarity of binary entity-mention vectors
- `collapse_by_simulation.png` — % of question-rounds where all 10 runs share one entity set (95% Wilson CI)

(Llama/Mistral sides and the `search` variant have no cross-model entity files yet, so they get text
metrics only.)

In [2]:
# ── DeepSeek-side entity diagrams (unique entities / entity similarity / collapse-by-simulation) ──
import re
from scipy.stats import binomtest

ENTITY_OUT = os.path.join(OUT_ROOT, "deepseek-r1-distill-qwen-7b")  # entity data is DeepSeek-side only

# Auto-discover the newest entity re-run dump in the repo root (gitignored; timestamp varies).
_PAT = re.compile(r"entity re-run for workshop paper-.*")
_outer = sorted(d for d in os.listdir(".") if _PAT.fullmatch(d) and os.path.isdir(d))
if not _outer:
    raise FileNotFoundError("No 'entity re-run for workshop paper-*' folder in the repo root.")
_inner = [d for d in os.listdir(_outer[-1]) if os.path.isdir(os.path.join(_outer[-1], d))]
ENTITY_BASE = os.path.join(_outer[-1], _inner[0]) if _inner else _outer[-1]
print("entity dump:", ENTITY_BASE)

CROSS_FILES = {
    "Replace All": (COLORS["Replace All"],
        "model_collapse_log_graphite_cross_model_replace_all_Qwen_Qwen2.5-14B-Instruct_local_replace_all.entities_by_round.jsonl"),
    "Replace One": (COLORS["Replace One"],
        "model_collapse_log_graphite_cross_model_replace_one_Qwen_Qwen2.5-14B-Instruct_local_replace_one.entities_by_round.jsonl"),
}

def _entity_similarity(vectors):
    n = len(vectors)
    if n < 2:
        return 0.0
    s = []
    for i in range(n):
        for j in range(i + 1, n):
            a, b = vectors[i], vectors[j]
            na, nb = np.linalg.norm(a), np.linalg.norm(b)
            s.append(float(np.dot(a, b) / (na * nb)) if na > 0 and nb > 0 else 0.0)
    return float(np.mean(s))

def analyze_entity_file(path):
    """Per-round mean unique-entity count and mean entity similarity (matches entity_extraction.py)."""
    pru, prs, mx = defaultdict(list), defaultdict(list), 0
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            rec = json.loads(line)
            rounds, counts, mapped = rec["round_numbers"], rec["entity_counts"], rec["mapped_entities"]
            mx = max(mx, len(rounds))
            for idx in range(len(rounds)):
                pru[idx].append(len(counts[idx]))
                runs = mapped[idx]
                vocab = sorted({e for run in runs for e in run})
                vi = {e: i for i, e in enumerate(vocab)}
                vecs = []
                for run in runs:
                    v = np.zeros(len(vocab))
                    for e in run:
                        v[vi[e]] = 1.0
                    vecs.append(v)
                prs[idx].append(_entity_similarity(vecs))
    return ([float(np.mean(pru[i])) for i in range(mx)], [float(np.mean(prs[i])) for i in range(mx)])

def _wilson(count, nobs):
    if nobs == 0:
        return 0.0, 0.0, 0.0
    count, nobs = int(round(count)), int(round(nobs))
    ci = binomtest(count, nobs).proportion_ci(confidence_level=0.95, method="wilson")
    return 100.0 * count / nobs, 100.0 * ci.low, 100.0 * ci.high

def collapse_metrics(path):
    """% of question-rounds collapsed (all 10 runs share one canonical entity set), with 95% Wilson CIs."""
    n = ns = ne = cr = tr = 0
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            rec = json.loads(line)
            mapped = rec["mapped_entities"]
            nr = len(rec["round_numbers"])
            flags = [len({frozenset(run) for run in mapped[i]}) == 1 for i in range(nr)]
            if not flags:
                continue
            n += 1; ns += int(flags[0]); ne += int(flags[-1]); cr += sum(flags); tr += len(flags)
    return {"start": _wilson(ns, n), "end": _wilson(ne, n), "rounds": _wilson(cr, tr)}

# Load + compute
ent = {}
for label, (color, fname) in CROSS_FILES.items():
    path = os.path.join(ENTITY_BASE, fname)
    if not os.path.exists(path):
        print("  MISSING:", fname); continue
    u, s = analyze_entity_file(path)
    ent[label] = {"color": color, "unique": u, "similarity": s, "collapse": collapse_metrics(path)}
    print(f"loaded {label}: {len(u)} rounds")
assert ent, "No cross-model entity files found in the dump."
os.makedirs(ENTITY_OUT, exist_ok=True)

# 1) + 2) unique entities and entity similarity (overlaid line charts)
for which, ylabel, title, fn in [
    ("unique", "Unique Entities", "Unique Entities Per Round", "unique_entities_per_round.png"),
    ("similarity", "Entity Similarity", "Entity Similarity Per Round", "entity_similarity_per_round.png")]:
    fig, ax = plt.subplots(figsize=(10, 6))
    for label, d in ent.items():
        vals = d[which]
        ax.plot(range(1, len(vals) + 1), vals, linewidth=2.5, label=label, color=d["color"])
    ax.set_xlabel("Round", fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(f"deepseek-r1-distill-qwen-7b — {title}", fontsize=13)
    ax.legend()
    fig.tight_layout()
    fig.savefig(os.path.join(ENTITY_OUT, fn), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("wrote", os.path.join(ENTITY_OUT, fn))

# 3) collapse by simulation (grouped bars, 95% Wilson CI)
sims = list(ent.keys())
x = np.arange(len(sims)); width = 0.25
fig, ax = plt.subplots(figsize=(8, 6))
for k, label, color, off in [
    ("start",  "% Collapsed at Start", "#1f77b4", -width),
    ("end",    "% Collapsed at End",   "#ff7f0e", 0.0),
    ("rounds", "% Rounds Collapsed",   "#2ca02c", width)]:
    p  = np.array([ent[s]["collapse"][k][0] for s in sims])
    lo = np.array([ent[s]["collapse"][k][1] for s in sims])
    hi = np.array([ent[s]["collapse"][k][2] for s in sims])
    yerr = np.vstack([p - lo, hi - p])
    bars = ax.bar(x + off, p, width, yerr=yerr, capsize=4, label=label, color=color)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f"{bar.get_height():.1f}", ha="center", va="bottom",
                fontsize=9, fontweight="bold", color=color)
ax.set_ylabel("%", fontsize=12)
ax.set_title("deepseek-r1-distill-qwen-7b — Collapse by Simulation (error bars: 95% Wilson CI)", fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels([f"{s.lower()}\nsimulation" for s in sims], fontsize=10)
ax.legend(loc="upper right")
ax.set_ylim(0, 115)
fig.tight_layout()
fig.savefig(os.path.join(ENTITY_OUT, "collapse_by_simulation.png"), dpi=150, bbox_inches="tight")
plt.close(fig)
print("wrote", os.path.join(ENTITY_OUT, "collapse_by_simulation.png"))

entity dump: entity re-run for workshop paper-20260630T060414Z-3-001\entity re-run for workshop paper


loaded Replace All: 10 rounds


loaded Replace One: 20 rounds


wrote cross_model_baseline_visualizations\deepseek-r1-distill-qwen-7b\unique_entities_per_round.png


wrote cross_model_baseline_visualizations\deepseek-r1-distill-qwen-7b\entity_similarity_per_round.png


wrote cross_model_baseline_visualizations\deepseek-r1-distill-qwen-7b\collapse_by_simulation.png
